In [1]:
"""
MAP Competition - Qwen3-4B Inference with LoRA Adapter
Single model inference for Kaggle submission with Explainable AI
"""

import os
import gc
import json
import re
import warnings
from collections import Counter
from pathlib import Path

warnings.filterwarnings('ignore')

def log(msg):
    print(msg, flush=True)

log("Starting imports...")

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoModelForCausalLM, AutoTokenizer, DataCollatorWithPadding
from peft import PeftModel, PeftConfig
from scipy.special import softmax
from tqdm import tqdm

# ============================================================================
# CONFIGURATION
# ============================================================================

TRAIN_PATH = "/kaggle/input/datasets/mkhanshovon43/map-ds/DS/train.csv"
TEST_PATH = "/kaggle/input/datasets/mkhanshovon43/map-ds/DS/test.csv"

# Qwen3-4B Model Paths - will be resolved by find functions
# Kaggle converts dataset names to lowercase with hyphens
BASE_MODEL_CANDIDATES = [
    "/kaggle/input/qwen3-4b-base",
    "/kaggle/input/qwen3-4b_base",
    "/kaggle/input/qwen3-4b-base/qwen3-4b",
    "/kaggle/input/qwen3-4b_base/qwen3-4b",
    "/kaggle/input/qwen34b-base",
    "/kaggle/input/datasets/mdmarufhasanrubab/qwen3-4b-base/qwen3-4b",
]

ADAPTER_CANDIDATES = [
    "/kaggle/input/map-qwen3-4b-trained",
    "/kaggle/input/mapqwen3-4b-trained",
    "/kaggle/input/map-qwen34b-trained",
    # Direct paths to adapter location (if dataset structure is preserved)
    "/kaggle/input/datasets/mkhanshovon43/map-qwen3-4b-trained/map_multi_llm_runs/Qwen3-4B-Instruct-2507/full_train/Qwen3-4B-Instruct-2507",
    "/kaggle/input/datasets/mkhanshovon43/map-qwen3-4b-trained/map_multi_llm_runs/Qwen3-4B-Instruct-2507/full_train/Qwen3-4B-Instruct-2507",
]

MAX_LEN = 256
BATCH_SIZE = 4

# Explainable AI Configuration - uses the SAME Qwen3-4B base model
USE_LLM_EXPLAINER = True
USE_LLM_VERIFIER = False
KB_BUILD_TOP_N = 50
KB_EXAMPLES_PER_MISCON = 3
KB_PATH = Path("misconception_kb.json")

# Device configuration (defined at module level for all cells)
device = "cuda:0" if torch.cuda.is_available() else "cpu"

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def find_model_path(base_paths):
    """Find the actual model directory from a list of candidate paths."""
    import glob
    
    # Handle single path or list of paths
    if isinstance(base_paths, str):
        base_paths = [base_paths]
    
    for base_path in base_paths:
        if not os.path.exists(base_path):
            continue
            
        # Check base path first
        if os.path.exists(os.path.join(base_path, "config.json")):
            # Verify it has model weights
            if (os.path.exists(os.path.join(base_path, "model.safetensors")) or
                os.path.exists(os.path.join(base_path, "pytorch_model.bin")) or
                glob.glob(os.path.join(base_path, "model*.safetensors"))):
                return base_path

        # Search for config.json with model weights in subdirectories
        for depth in range(1, 5):
            pattern = os.path.join(base_path, "/".join(["*"] * depth), "config.json")
            matches = glob.glob(pattern)
            for m in matches:
                dir_path = os.path.dirname(m)
                has_weights = (os.path.exists(os.path.join(dir_path, "model.safetensors")) or
                              os.path.exists(os.path.join(dir_path, "pytorch_model.bin")) or
                              glob.glob(os.path.join(dir_path, "model*.safetensors")))
                if has_weights:
                    return dir_path
    
    # Return first candidate as fallback
    return base_paths[0] if base_paths else None


def find_adapter_path(base_paths):
    """Find the actual adapter directory from a list of candidate paths."""
    import glob
    
    # Handle single path or list of paths
    if isinstance(base_paths, str):
        base_paths = [base_paths]
    
    for base_path in base_paths:
        if not os.path.exists(base_path):
            continue
            
        # Check base path first
        if os.path.exists(os.path.join(base_path, "adapter_config.json")):
            return base_path

        # Search for adapter_config.json in subdirectories (up to 6 levels deep)
        for depth in range(1, 7):
            pattern = os.path.join(base_path, "/".join(["*"] * depth), "adapter_config.json")
            matches = glob.glob(pattern)
            for m in matches:
                return os.path.dirname(m)
    
    # Return first candidate as fallback
    return base_paths[0] if base_paths else None


def list_kaggle_inputs():
    """List all available Kaggle input directories for debugging."""
    input_dir = "/kaggle/input"
    if os.path.exists(input_dir):
        dirs = os.listdir(input_dir)
        log(f"Available Kaggle inputs: {dirs}")
        # Also list subdirectories for each input
        for d in dirs:
            subpath = os.path.join(input_dir, d)
            if os.path.isdir(subpath):
                try:
                    subdirs = os.listdir(subpath)[:5]
                    log(f"  {d}/: {subdirs}")
                except:
                    pass
        return dirs
    return []


def format_input(row):
    """Format input text for Qwen3-4B model."""
    x = "This answer is correct." if row['is_correct'] else "This answer is incorrect."
    return (
        f"Question: {row['QuestionText']}\n"
        f"Answer: {row['MC_Answer']}\n"
        f"{x}\n"
        f"Student Explanation: {row['StudentExplanation']}"
    )


# ============================================================================
# MAIN INFERENCE
# ============================================================================

log("="*60)
log("MAP COMPETITION - QWEN3-4B INFERENCE (with LoRA)")
log("="*60)

if torch.cuda.is_available():
    log(f"GPU: {torch.cuda.get_device_name(0)}")

# List available inputs for debugging
available_inputs = list_kaggle_inputs()

# Load data
log("Loading data...")
train_df = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

log(f"Train: {train_df.shape}, Test: {test.shape}")

# Prepare labels
train_df['Misconception'] = train_df['Misconception'].fillna('NA')
train_df['target'] = train_df['Category'] + ':' + train_df['Misconception']
train_df['label_str'] = train_df['target']  # Alias for explainability

le = LabelEncoder()
le.fit(train_df['target'])
n_classes = len(le.classes_)
log(f"Number of classes: {n_classes}")

# Find correct answers for is_correct flag
idx = train_df['Category'].apply(lambda c: c.split('_')[0]) == 'True'
correct = train_df.loc[idx].copy()
correct['c'] = correct.groupby(['QuestionId', 'MC_Answer'])['MC_Answer'].transform('count')
correct = correct.sort_values('c', ascending=False).drop_duplicates(['QuestionId'])
correct = correct[['QuestionId', 'MC_Answer']]
correct['is_correct'] = 1

# Build correct_by_q_full dictionary for explainability
correct_by_q_full = dict(zip(correct['QuestionId'], correct['MC_Answer']))

# Merge is_correct into test
test = test.merge(correct, on=['QuestionId', 'MC_Answer'], how='left')
test['is_correct'] = test['is_correct'].fillna(0)

# Merge is_correct into train_df for explainability
train_df = train_df.merge(correct[['QuestionId', 'MC_Answer', 'is_correct']], 
                           on=['QuestionId', 'MC_Answer'], how='left')
train_df['is_correct'] = train_df['is_correct'].fillna(0)

# Format input text
test['text'] = test.apply(format_input, axis=1)

# Build family map for filtering
fam_map = {}
for _, row in test.iterrows():
    fam_map[row['row_id']] = 'True_' if row['is_correct'] == 1 else 'False_'

# Find model paths - search through candidates
log("Searching for base model...")
base_model_path = find_model_path(BASE_MODEL_CANDIDATES)
log(f"Found base model at: {base_model_path}")

log("Searching for adapter...")
adapter_path = find_adapter_path(ADAPTER_CANDIDATES)
log(f"Found adapter at: {adapter_path}")

# Verify paths exist
if not base_model_path or not os.path.exists(base_model_path):
    log("ERROR: Could not find base model!")
    log("Please check your Kaggle dataset names above and update BASE_MODEL_CANDIDATES")
    raise FileNotFoundError(f"Base model path not found. Searched: {BASE_MODEL_CANDIDATES}")

if not adapter_path or not os.path.exists(adapter_path):
    log("ERROR: Could not find adapter!")
    log("Please check your Kaggle dataset names above and update ADAPTER_CANDIDATES")
    raise FileNotFoundError(f"Adapter path not found. Searched: {ADAPTER_CANDIDATES}")

# Store base model path globally for explainer
BASE_MODEL_PATH_RESOLVED = base_model_path

# Load tokenizer from base model (use local_files_only=True for local paths)
log("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    base_model_path, 
    trust_remote_code=True,
    local_files_only=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load base model for classification
log("Loading Qwen3-4B base model...")
base_model = AutoModelForSequenceClassification.from_pretrained(
    base_model_path,
    num_labels=n_classes,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map=device,
    local_files_only=True
)
base_model.config.pad_token_id = tokenizer.pad_token_id

# Load LoRA adapter
log("Loading LoRA adapter...")
model = PeftModel.from_pretrained(base_model, adapter_path, local_files_only=True)
model.eval()
log("Model with LoRA adapter loaded!")

# Tokenize
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

ds_test = Dataset.from_pandas(test[['text']])
ds_test = ds_test.map(tokenize, batched=True, remove_columns=['text'])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")
dataloader = DataLoader(ds_test, batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator)

# Run inference
log(f"Running inference on {len(ds_test)} samples...")
all_logits = []
with torch.no_grad():
    for batch in tqdm(dataloader, desc="Inference"):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        all_logits.append(outputs.logits.float().cpu().numpy())

predictions = np.concatenate(all_logits, axis=0)
probs = softmax(predictions, axis=1)
top_indices = np.argsort(-probs, axis=1)

# Decode labels
flat_indices = top_indices.flatten()
decoded_labels = le.inverse_transform(flat_indices)
top_labels = decoded_labels.reshape(top_indices.shape)

# Apply family filter and create submission
log("Creating submission with family filtering...")
final_predictions = []

for i, row_id in enumerate(test.row_id.values):
    pref = fam_map.get(row_id, 'False_')

    # Filter predictions by family prefix
    filtered = [lbl for lbl in top_labels[i] if lbl.startswith(pref)]

    # Take top 3
    top3 = filtered[:3]

    # Fill if needed
    while len(top3) < 3:
        filler = f"{pref}Neither:NA"
        if filler not in top3:
            top3.append(filler)
        else:
            top3.append(f"{pref}Correct:NA" if pref == "True_" else f"{pref}Misconception:NA")

    final_predictions.append(" ".join(top3[:3]))

# Save submission
submission = pd.DataFrame({
    'row_id': test.row_id.values,
    'Category:Misconception': final_predictions
})
submission.to_csv('submission.csv', index=False)

log("\n" + "="*60)
log("SUBMISSION SAVED: submission.csv")
log("="*60)
log(str(submission.head(10)))
log(f"\nTotal predictions: {len(submission)}")

# Clean up classification model to free memory for CausalLM explainer
log("\nCleaning up classification model for explainer...")
del model
del base_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Starting imports...
MAP COMPETITION - QWEN3-4B INFERENCE (with LoRA)
GPU: NVIDIA H100 80GB HBM3
Available Kaggle inputs: ['competitions', 'datasets']
  competitions/: ['ai-mathematical-olympiad-progress-prize-3']
  datasets/: ['mkhanshovon43', 'mdmarufhasanrubab']
Loading data...
Train: (36696, 7), Test: (3, 5)
Number of classes: 65
Searching for base model...
Found base model at: /kaggle/input/datasets/mdmarufhasanrubab/qwen3-4b-base/qwen3-4b
Searching for adapter...
Found adapter at: /kaggle/input/datasets/mkhanshovon43/map-qwen3-4b-trained/map_multi_llm_runs/Qwen3-4B-Instruct-2507/full_train/Qwen3-4B-Instruct-2507
Loading tokenizer...
Loading Qwen3-4B base model...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Qwen3ForSequenceClassification LOAD REPORT from: /kaggle/input/datasets/mdmarufhasanrubab/qwen3-4b-base/qwen3-4b
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading LoRA adapter...
Model with LoRA adapter loaded!


Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Running inference on 3 samples...


Inference: 100%|██████████| 1/1 [00:01<00:00,  1.25s/it]

Creating submission with family filtering...

SUBMISSION SAVED: submission.csv
   row_id                             Category:Misconception
0   36696  True_Correct:NA True_Neither:NA True_Misconcep...
1   36697  False_Misconception:WNB False_Neither:NA False...
2   36698  True_Neither:NA True_Correct:NA True_Misconcep...

Total predictions: 3

Cleaning up classification model for explainer...


## Cell 8 — Explainable AI: Misconception Knowledge Base

A key differentiator of this pipeline is **explainability**. We build a **Misconception Knowledge Base (KB)** using the **same Qwen3-4B base model**:

1. For each misconception type (e.g., "Additive", "Duplication"), sample representative student explanations.
2. Load the Qwen3-4B base model from `BASE_MODEL_PATH_RESOLVED` as a CausalLM (for text generation).
3. Prompt it to generate a structured "knowledge card":
   - Plain definition
   - Typical wrong thought patterns
   - Correct rule
   - Fix steps for teachers
   - Quick self-check question
4. Cache the KB as JSON for reuse.

This enables the pipeline to not only **predict** misconceptions but also **explain** them to teachers — using a single model family throughout.

In [2]:
# ============================================================
# EXPLAINABLE AI — MISCONCEPTION KNOWLEDGE BASE (CORRECTED)
# Uses the SAME Qwen3-4B base model (loaded as CausalLM)
# ============================================================

import os
import gc
import json
import re
from collections import Counter
from pathlib import Path

import torch
import pandas as pd
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer

# ----------------------------------------------------------------
# CONFIG DEFAULTS (safe fallbacks if not defined earlier)
# ----------------------------------------------------------------
USE_LLM_EXPLAINER = globals().get("USE_LLM_EXPLAINER", True)
USE_LLM_VERIFIER = globals().get("USE_LLM_VERIFIER", False)
KB_BUILD_TOP_N = globals().get("KB_BUILD_TOP_N", 65)
KB_EXAMPLES_PER_MISCON = globals().get("KB_EXAMPLES_PER_MISCON", 3)
KB_PATH = Path(globals().get("KB_PATH", "./misconception_kb.json"))

# ----------------------------------------------------------------
# HELPERS
# ----------------------------------------------------------------
def safe_str(x):
    if pd.isna(x):
        return ""
    return str(x)


def ensure_pad_token(tok):
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok


def get_model_device(model):
    try:
        return next(model.parameters()).device
    except StopIteration:
        return torch.device("cpu")


def extract_first_json_object(text: str):
    """
    Safer JSON extraction than greedy regex.
    Tries fenced json first, then scans braces.
    """
    if not text:
        return None

    fenced = re.search(r"```json\s*(\{.*?\})\s*```", text, flags=re.S)
    if fenced:
        candidate = fenced.group(1)
        try:
            return json.loads(candidate)
        except Exception:
            pass

    start = text.find("{")
    if start == -1:
        return None

    depth = 0
    for i in range(start, len(text)):
        ch = text[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                candidate = text[start:i + 1]
                try:
                    return json.loads(candidate)
                except Exception:
                    return None
    return None


def validate_kb_card(card):
    if not isinstance(card, dict):
        return False

    required = [
        "plain_definition",
        "typical_wrong_thought",
        "correct_rule",
        "fix_steps",
        "quick_check",
        "tiny_example",
    ]
    for k in required:
        if k not in card:
            return False

    return (
        isinstance(card["plain_definition"], str)
        and isinstance(card["typical_wrong_thought"], list)
        and isinstance(card["correct_rule"], list)
        and isinstance(card["fix_steps"], list)
        and isinstance(card["quick_check"], str)
        and isinstance(card["tiny_example"], str)
    )


def fallback_kb_card(miscon_name: str):
    return {
        "plain_definition": f"This misconception is about: {miscon_name}. The student may be using an incomplete or incorrect rule.",
        "typical_wrong_thought": [
            "A shortcut or remembered pattern is being applied in the wrong way."
        ],
        "correct_rule": [
            "Use the exact mathematical rule, not a guessed shortcut.",
            "Check the reasoning step by step."
        ],
        "fix_steps": [
            "Read what the question is asking.",
            "Write the correct rule or definition.",
            "Apply it one step at a time.",
            "Check the result with an easier example."
        ],
        "quick_check": "Would the same method still work on a very simple example?",
        "tiny_example": "Try the same method on small easy numbers and see whether it still makes sense."
    }


def parse_top3_predictions(value):
    """
    Accepts:
    - Python list
    - JSON string list
    - delimiter-based string
    - single label string
    Returns list[str]
    """
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []

    if isinstance(value, list):
        return [safe_str(x).strip() for x in value if safe_str(x).strip()]

    s = safe_str(value).strip()
    if not s:
        return []

    if s.startswith("[") and s.endswith("]"):
        try:
            arr = json.loads(s)
            if isinstance(arr, list):
                return [safe_str(x).strip() for x in arr if safe_str(x).strip()]
        except Exception:
            pass

    for delim in [" ||| ", "|||", " | ", "\t", "\n", "; "]:
        if delim in s:
            return [x.strip() for x in s.split(delim) if x.strip()]

    return [s]


def normalize_pred_storage_column(df: pd.DataFrame, pred_col="pred_top3"):
    if pred_col not in df.columns:
        return df
    df = df.copy()

    def _norm(x):
        if isinstance(x, list):
            return json.dumps(x, ensure_ascii=False)
        return x

    df[pred_col] = df[pred_col].apply(_norm)
    return df

# ----------------------------------------------------------------
# LOAD EXPLAINER MODEL
# ----------------------------------------------------------------
def load_explainer():
    """
    Load the SAME Qwen3-4B base model as CausalLM for text generation.
    Tries 4-bit first on GPU, then float16, then CPU float32.
    """
    if not USE_LLM_EXPLAINER:
        return None, None

    print(f"Loading explainer from: {BASE_MODEL_PATH_RESOLVED}")

    if not os.path.exists(BASE_MODEL_PATH_RESOLVED):
        print(f"ERROR: Base model path not found: {BASE_MODEL_PATH_RESOLVED}")
        return None, None

    tok = AutoTokenizer.from_pretrained(
        BASE_MODEL_PATH_RESOLVED,
        trust_remote_code=True,
        local_files_only=True,
    )
    tok = ensure_pad_token(tok)

    if "cuda" in str(device).lower():
        try:
            from transformers import BitsAndBytesConfig
            bnb_cfg = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
            )
            mdl = AutoModelForCausalLM.from_pretrained(
                BASE_MODEL_PATH_RESOLVED,
                quantization_config=bnb_cfg,
                device_map="auto",
                torch_dtype=torch.float16,
                trust_remote_code=True,
                local_files_only=True,
            )
            mdl.eval()
            print("Explainer loaded in 4-bit.")
            return mdl, tok
        except Exception as e:
            print(f"4-bit load failed: {str(e)[:120]}")

    if "cuda" in str(device).lower():
        try:
            mdl = AutoModelForCausalLM.from_pretrained(
                BASE_MODEL_PATH_RESOLVED,
                device_map="auto",
                torch_dtype=torch.float16,
                trust_remote_code=True,
                local_files_only=True,
            )
            mdl.eval()
            print("Explainer loaded in float16.")
            return mdl, tok
        except Exception as e:
            print(f"float16 load failed: {str(e)[:120]}")

    try:
        mdl = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_PATH_RESOLVED,
            torch_dtype=torch.float32,
            trust_remote_code=True,
            low_cpu_mem_usage=True,
            local_files_only=True,
        )
        mdl.eval()
        print("Explainer loaded on CPU.")
        return mdl, tok
    except Exception as e:
        print(f"All explainer loading methods failed: {e}")
        return None, None


def llm_generate(prompt: str, max_new_tokens=320, explainer_mdl=None, explainer_tok=None):
    """Safer generation helper."""
    if explainer_mdl is None or explainer_tok is None:
        return ""

    messages = [{"role": "user", "content": prompt}]
    try:
        chat_prompt = explainer_tok.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        chat_prompt = explainer_tok.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    inputs = explainer_tok(
        chat_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048,
    )

    try:
        model_device = get_model_device(explainer_mdl)
        inputs = {k: v.to(model_device) for k, v in inputs.items()}
    except Exception:
        pass

    with torch.no_grad():
        out = explainer_mdl.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            pad_token_id=explainer_tok.eos_token_id,
        )

    gen_ids = out[0][inputs["input_ids"].shape[1]:]
    text = explainer_tok.decode(gen_ids, skip_special_tokens=True).strip()
    return text

# ----------------------------------------------------------------
# LOAD EXPLAINER + BUILD / LOAD MISCONCEPTION KB
# ----------------------------------------------------------------
explainer_model, explainer_tok = load_explainer()


def build_misconception_kb(train_df, explainer_model, explainer_tok,
                           kb_path=KB_PATH,
                           top_n=KB_BUILD_TOP_N,
                           examples_per_miscon=KB_EXAMPLES_PER_MISCON):
    df = train_df.copy()
    df["miscon_only"] = df["Misconception"].astype(str)

    freq = Counter(df["miscon_only"].tolist())
    top_miscons = [m for m, _ in freq.most_common(top_n) if m != "NA"]

    print(f"Preparing KB for {len(top_miscons)} misconception types...")

    if kb_path.exists():
        try:
            kb = json.loads(kb_path.read_text(encoding="utf-8"))
            if isinstance(kb, dict) and len(kb) > 0:
                print(f"Loaded cached KB: {len(kb)} entries")
                return kb
        except Exception:
            print("Existing KB file could not be read. Rebuilding...")

    kb = {}

    for i, miscon in enumerate(top_miscons, start=1):
        sub = df[df["miscon_only"] == miscon]
        sample = sub.sample(n=min(examples_per_miscon, len(sub)), random_state=42)

        examples = []
        for _, r in sample.iterrows():
            examples.append({
                "question": safe_str(r.get("QuestionText"))[:240],
                "student_answer": safe_str(r.get("MC_Answer"))[:80],
                "student_explanation": safe_str(r.get("StudentExplanation"))[:240],
            })

        card = None

        if USE_LLM_EXPLAINER and explainer_model is not None:
            prompt = f"""
You are a math teacher creating a misconception knowledge card.

Misconception name:
{miscon}

Student examples:
{json.dumps(examples, ensure_ascii=False, indent=2)}

Return ONLY valid JSON.
Use exactly these keys:
- plain_definition
- typical_wrong_thought
- correct_rule
- fix_steps
- quick_check
- tiny_example

Requirements:
- plain_definition: 1-2 short sentences
- typical_wrong_thought: list of 2-4 short strings
- correct_rule: list of 2-4 short strings
- fix_steps: list of 4-6 short steps
- quick_check: 1 short question
- tiny_example: 2-4 short lines
- Do not include markdown
- Do not include extra keys
""".strip()

            out = llm_generate(
                prompt,
                max_new_tokens=420,
                explainer_mdl=explainer_model,
                explainer_tok=explainer_tok,
            )

            card = extract_first_json_object(out)
            if not validate_kb_card(card):
                card = None

        if card is None:
            card = fallback_kb_card(miscon)

        kb[miscon] = card

        if i % 10 == 0 or i == len(top_miscons):
            print(f"  Built {i}/{len(top_miscons)} cards")

    kb_path.write_text(json.dumps(kb, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Saved KB to {kb_path}")
    return kb


kb = build_misconception_kb(
    train_df=train_df,
    explainer_model=explainer_model,
    explainer_tok=explainer_tok,
    kb_path=KB_PATH,
    top_n=KB_BUILD_TOP_N,
    examples_per_miscon=KB_EXAMPLES_PER_MISCON,
)

if len(kb) > 0:
    k0 = next(iter(kb))
    print("\nSample KB entry:")
    print(k0)
    print(json.dumps(kb[k0], ensure_ascii=False, indent=2)[:800])
else:
    print("No KB entries built.")



Loading Qwen3-4B explainer from: /kaggle/input/datasets/mdmarufhasanrubab/qwen3-4b-base/qwen3-4b
4-bit quantization failed: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`
Trying float16 without quantization...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Successfully loaded Qwen3-4B explainer (float16, no quantization)

Building KB for 35 misconception types...
  Built 20/35 cards...
Saved KB: 35 entries to misconception_kb.json

--- Sample KB Card: 'Incomplete' ---
{
  "plain_definition": "Incomplete refers to a misunderstanding where a student provides an answer that is partially correct but fails to fully simplify the fraction or recognize the correct numerator and denominator.",
  "typical_wrong_thought": [
    "Only counting the unshaded parts without considering the total number of parts",
    "Failing to simplify the fraction to its lowest terms",
    "Assuming the total number of parts is incorrect",
    "Not recognizing that the shaded and unshaded parts should both be considered in the fraction"
  ],
  "correct_rule": [
    "To find the fraction of the shape that is not shaded, count the number of unshaded parts and divide by the total number of parts.",
    "Simplify the resulting fraction to its lowest terms by dividing bot

## Cell 9 — Explainable AI: Teacher Feedback Generator

The final explainability component generates **personalized teacher feedback** for each student using the **same Qwen3-4B base model**:
- Uses the Qwen3-4B prediction (with LoRA) + KB card to produce targeted feedback
- Includes: what the student seems to think, the misconception in simple words, step-by-step correct logic, and a self-check question
- Rule-based verification ensures consistency (e.g., don't say "wrong" if the student was correct)
- Optional Qwen3-4B-based verification to fix contradictions

**Single Model Family Pipeline**: Classification uses Qwen3-4B + LoRA adapter, explanation generation uses Qwen3-4B base model.

In [3]:
# ============================================================
# EXPLAINABLE AI — TEACHER FEEDBACK GENERATOR (CORRECTED)
# Uses the SAME Qwen3-4B base model loaded above
# ============================================================

def get_correct_choice(qid):
    return safe_str(correct_by_q_full.get(qid, ""))


def get_kb_card_for_label(pred_label: str, kb: dict):
    """
    If pred_label is 'Category:Misconception', retrieve by misconception part.
    """
    pred_label = safe_str(pred_label).strip()
    miscon = pred_label.split(":", 1)[1].strip() if ":" in pred_label else pred_label
    if miscon in kb:
        return miscon, kb[miscon]
    return miscon, fallback_kb_card(miscon if miscon else "Unknown misconception")


def rule_based_verify(feedback: str, is_correct: int, correct_choice: str) -> str:
    txt = safe_str(feedback).strip()

    if is_correct == 1:
        txt = re.sub(r"\bincorrect\b", "not quite right", txt, flags=re.I)
        txt = re.sub(r"\bwrong\b", "not quite right", txt, flags=re.I)

    if is_correct == 0 and correct_choice:
        header = f"Correct answer: {correct_choice}"
        if header not in txt:
            txt = header + "\n\n" + txt

    return txt.strip()


def llm_verify(feedback: str, explainer_model=None, explainer_tok=None) -> str:
    """
    Optional second pass. Kept constrained.
    """
    if not USE_LLM_VERIFIER or explainer_model is None:
        return feedback

    prompt = f"""
Edit the following teacher feedback for clarity only.

Rules:
- Do not change the mathematical meaning
- Do not add new facts
- Keep it concise
- Keep the same structure

Text:
{feedback}
""".strip()

    out = llm_generate(
        prompt,
        max_new_tokens=220,
        explainer_mdl=explainer_model,
        explainer_tok=explainer_tok,
    )
    return out if out else feedback


def generate_teacher_feedback(row, pred_label: str, pred_top3: list, kb: dict,
                              explainer_model=None, explainer_tok=None) -> str:
    qtext   = safe_str(row.get("QuestionText"))[:400]
    ans     = safe_str(row.get("MC_Answer"))[:120]
    expl    = safe_str(row.get("StudentExplanation"))[:400]
    qid     = row.get("QuestionId")
    correct = get_correct_choice(qid)

    try:
        ic = int(row.get("is_correct", -1))
    except Exception:
        ic = -1

    ic_str = {1: "Yes", 0: "No"}.get(ic, "Unknown")

    miscon_name, card = get_kb_card_for_label(pred_label, kb)

    if USE_LLM_EXPLAINER and explainer_model is not None:
        prompt = f"""
You are a kind and clear math teacher.

Use ONLY the information provided below.
Do not invent extra facts.
If something is uncertain, stay general.

Question:
{qtext}

Student chose:
{ans}

Is the student's choice correct?
{ic_str}

Correct choice:
{correct}

Student explanation:
{expl}

Top predicted misconceptions:
{json.dumps(pred_top3, ensure_ascii=False)}

Main predicted misconception:
{pred_label}

Retrieved misconception knowledge card:
{json.dumps(card, ensure_ascii=False, indent=2)}

Write teacher feedback in exactly this format:

Student thinking:
1-2 short lines explaining what the student seems to be thinking.

Misconception:
1 short line in simple words.

Fix steps:
1. ...
2. ...
3. ...
4. ...
(you may add up to 6 steps total)

Self-check:
1 short question.

Style rules:
- encouraging
- simple
- concise
- mathematically careful
- no markdown code block
""".strip()

        fb = llm_generate(
            prompt,
            max_new_tokens=320,
            explainer_mdl=explainer_model,
            explainer_tok=explainer_tok,
        )
        if not fb:
            fb = "Student thinking:\nThe student may be applying an incomplete rule.\n\nMisconception:\nThe rule is being used in the wrong way.\n\nFix steps:\n1. Re-read the question.\n2. Write the correct rule.\n3. Apply it step by step.\n4. Check with an easier example.\n\nSelf-check:\nWould the same method work on a simpler case?"
    else:
        fb = f"""Student thinking:
The student may be using an incomplete or incorrect rule.

Misconception:
{miscon_name}

Fix steps:
1. Re-read the question carefully.
2. Write the correct mathematical rule.
3. Apply it one step at a time.
4. Check the result with an easier example.

Self-check:
Would the same method still work on a simpler example?
"""

    fb = rule_based_verify(fb, ic, correct)
    fb = llm_verify(fb, explainer_model=explainer_model, explainer_tok=explainer_tok)
    return fb.strip()


print("=" * 80)
print("EXPLAINABLE AI — SAMPLE TEACHER FEEDBACK (KB + PROMPT)")
print("=" * 80)

# Important fix:
# Do NOT leak ground-truth labels into predictions.
if "pred_top3" not in train_df.columns:
    print("WARNING: 'pred_top3' column not found. Demo will skip rows without predictions.")

demo_df = train_df.copy()

if "pred_top3" in demo_df.columns:
    demo_df = normalize_pred_storage_column(demo_df, pred_col="pred_top3")

rng = np.random.default_rng(42)
demo_idxs = rng.choice(len(demo_df), size=min(5, len(demo_df)), replace=False)

for idx in demo_idxs:
    row = demo_df.iloc[idx]

    pred_value = row.get("pred_top3", None)
    top3 = parse_top3_predictions(pred_value)

    if len(top3) == 0:
        print(f"\n{'─' * 80}")
        print(f"ROW {idx}")
        print("Skipped: no pred_top3 available for this row.")
        continue

    pred1 = top3[0]

    print(f"\n{'─' * 80}")
    print(f"ROW {idx}")
    print(f"{'─' * 80}")
    print(f"Question: {safe_str(row.get('QuestionText'))[:150]}")
    print(f"Student Answer: {safe_str(row.get('MC_Answer'))}")
    print(f"Student Explanation: {safe_str(row.get('StudentExplanation'))[:150]}")
    print(f"True Label: {safe_str(row.get('label_str'))}")
    print(f"Predictions: {top3}")

    correct_marker = "CORRECT" if (safe_str(row.get("label_str")) == pred1) else "WRONG"
    print(f"Top-1 Match: {correct_marker}")

    print("\n--- Teacher Feedback ---")
    feedback = generate_teacher_feedback(
        row=row,
        pred_label=pred1,
        pred_top3=top3,
        kb=kb,
        explainer_model=explainer_model,
        explainer_tok=explainer_tok,
    )
    print(feedback)
    print()

# Optional cleanup (leave commented if you still need the explainer afterward)
# if explainer_model is not None:
#     del explainer_model
#     del explainer_tok
#     gc.collect()
#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()
#     print("Explainer model cleaned up.")



  EXPLAINABLE AI — SAMPLE TEACHER FEEDBACK (Qwen3-4B)

────────────────────────────────────────────────────────────────────────────────
ROW 28398
────────────────────────────────────────────────────────────────────────────────
Question: This is part of a regular polygon. How many sides does it have? [Image: A diagram showing an obtuse angle labelled 144 degrees]
Student Answer: Not enough information
Student Explanation: it’s only showing a bit of the shape
True Label: False_Misconception:Unknowable
Predictions: ['False_Misconception:Unknowable']
Top-1 Match: CORRECT

--- Teacher Feedback (Qwen3-4B) ---
Correct answer: \( 10 \)

The student thinks the diagram doesn't show enough of the polygon to determine the number of sides.  
Misconception: They believe the problem can't be solved with the given info.  

1. In a regular polygon, each interior angle is equal.  
2. The formula for each interior angle is $ \frac{(n-2) \times 180}{n} $.  
3. Plug in 144° for the angle: $ \frac{(n-2) \ti